# 05 - Train the v3 Standard-KD multimodal student

Runs the exact frozen v3 Standard KD: equal-weight normalized audio/text teacher soft targets, temperature 2.0, KD weight 0.5.

Notebook 05 is backward-compatible with exact legacy text-teacher outputs: it can add the explicit epoch-0 selection metadata/alias without rerunning teacher inference. It also validates both selected teachers, the exact audio branch, and No-KD before Standard KD starts.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/multimodal-depression-kd'
CORE_COMMIT = '0d6ac621be7da1db7c1d769d075d142d0a740fd8'
CODE = Path('/content/multimodal-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
import pandas as pd
from datetime import datetime, timezone

TEXT=RUN/'03_student_text'
AUDIO=RUN/'03_student_audio'
TTEACH=RUN/'01_text_teacher'
ATEACH=RUN/'02_audio_teacher'
NOKD=RUN/'04_no_kd'
OUT=RUN/'05_standard_kd'
for p in [TEXT,AUDIO,TTEACH,ATEACH,NOKD]:
    assert p.is_dir(),p

# Self-heal a stale Colab code checkout before Standard KD.
def ensure_pinned_core():
    try:
        current=subprocess.check_output(
            ['git','-C',str(CODE),'rev-parse','HEAD'],text=True,
            stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        current=None

    if current!=CORE_COMMIT:
        print('Refreshing stale code checkout:',current,'->',CORE_COMMIT)
        if CODE.exists(): shutil.rmtree(CODE)
        CODE.mkdir(parents=True)
        subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'remote','add','origin',
                        f'https://github.com/{REPO}.git'],check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],
                       check=True,env=env)
        subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],
                       check=True,env=env)

    current=subprocess.check_output(
        ['git','-C',str(CODE),'rev-parse','HEAD'],text=True
    ).strip()
    assert current==CORE_COMMIT,(current,CORE_COMMIT)
    print('Verified Standard-KD core:',current)

ensure_pinned_core()

# ------------------------------------------------------------------
# Upgrade a legacy-but-exact text-teacher folder without recomputation.
# The current run may predate teacher_selection.json / selected_* aliases.
# ------------------------------------------------------------------
ta=json.loads((TTEACH/'text_target_audit.json').read_text())
tm=ta['dev34']
legacy_text_targets=TTEACH/'train_text_kd_targets.csv'
assert ta['train_participants']==107 and ta['dev_participants']==34
assert ta['test_opened'] is False
assert round(float(tm['macro_f1']),10)==round(0.8418604651162791,10)
assert tm['confusion_matrix']==[[19,4],[1,10]]

tx=pd.read_csv(legacy_text_targets)
assert len(tx)==107 and tx['participant_id'].nunique()==107
for col in ['participant_id','label','text_probability','text_logit']:
    assert col in tx.columns,col

selected_text_targets=TTEACH/'selected_train_kd_targets.csv'
if not selected_text_targets.is_file():
    shutil.copy2(legacy_text_targets,selected_text_targets)
    print('UPGRADE: created selected_train_kd_targets.csv from validated epoch-0 text targets.')

text_selection=TTEACH/'teacher_selection.json'
if not text_selection.is_file():
    selection={
        'teacher':'IDIAP participant InducT-GCN top250',
        'epoch0_definition':'original published checkpoint before any local fine-tuning',
        'epoch0_source_commit':'de8f477619ce61a2a65640a0735aa8dfe659ac59',
        'epoch0_dev34':tm,
        'selection_rule':'start with epoch 0; replace only on strict lexicographic DEV improvement: macro-F1, depressed-F1, AUROC',
        'fine_tuning_attempted':False,
        'selected_epoch':0,
        'selected_source':'original published checkpoint',
        'selected_targets':'selected_train_kd_targets.csv',
        'legacy_run_metadata_upgrade':True,
        'test_opened':False,
    }
    text_selection.write_text(json.dumps(selection,indent=2)+'\n')
    print('UPGRADE: created teacher_selection.json; selected text teacher = original epoch 0.')

ts=json.loads(text_selection.read_text())
assert ts['selected_epoch']==0
assert 'original' in ts['selected_source'].lower()
assert ts['test_opened'] is False
assert selected_text_targets.is_file()

# Audio teacher must already be explicit epoch-0 selection.
aas=json.loads((ATEACH/'teacher_selection.json').read_text())
assert aas['selected_epoch']==0,aas
assert 'original' in aas['selected_source'].lower(),aas
assert aas['test_opened'] is False
assert (ATEACH/'selected_train_kd_targets.csv').is_file()

# Student branches and No-KD must be the frozen exact-v3 sources.
am=json.loads((AUDIO/'metrics.json').read_text())
ac=json.loads((AUDIO/'complete.json').read_text())
ap=am['protocol']; av=am['dev34']
assert ac['status']=='PASS'
assert ap['batch_size']==20 and ap['crop_seed']==1300
assert ap['requested_epochs']==100 and ap['patience']==20
assert ap['completed_epochs']==71 and ap['best_epoch']==51
assert round(float(av['majority_vote']['macro_f1']),4)==0.6222
assert round(float(av['majority_vote']['depressed_f1']),4)==0.4444
assert round(float(av['auroc_soft_mean_probability']),4)==0.5257

nm=json.loads((NOKD/'metrics.json').read_text()); nv=nm['dev34']
assert nm['protocol']['test_opened'] is False
assert round(float(nv['macro_f1']),4)==0.7043
assert round(float(nv['depressed_f1']),4)==0.6087
assert round(float(nv['auroc']),4)==0.7470

print('PRECHECK PASS:')
print('  text teacher  -> original epoch 0')
print('  audio teacher -> original epoch 0')
print('  audio branch  -> exact v3 epoch 51')
print('  No-KD         -> exact frozen DEV-34 result')

# Preserve any non-empty failed/partial KD output before retrying.
if OUT.exists() and any(OUT.iterdir()) and not (OUT/'metrics.json').is_file():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_UTC')
    dst=OUT.with_name(OUT.name+'_attempt_'+stamp)
    shutil.move(str(OUT),str(dst))
    print('Preserved previous KD attempt:',dst)

if (OUT/'metrics.json').is_file():
    existing=json.loads((OUT/'metrics.json').read_text())
    em=existing.get('dev34',{})
    exact_existing=(
        round(float(em.get('macro_f1',-1)),4)==0.7236 and
        round(float(em.get('depressed_f1',-1)),4)==0.6667 and
        round(float(em.get('auroc',-1)),4)==0.7668
    )
else:
    exact_existing=False

if exact_existing:
    print('REUSE: exact Standard-KD result already present.')
else:
    cmd=[sys.executable,'-m','scripts.students.participant_student_v3.train_fusion_standard_kd',
         '--text-branch',str(TEXT),'--audio-branch',str(AUDIO),
         '--teacher-text',str(TTEACH),'--teacher-audio',str(ATEACH),
         '--no-kd-reference',str(NOKD),'--output',str(OUT),
         '--seed','103','--epochs','80','--patience','20',
         '--temperature','2.0','--kd-weight','0.5']

    print('Starting Standard KD...')
    proc=subprocess.run(cmd,cwd=CODE,text=True)
    if proc.returncode!=0:
        raise RuntimeError(
            f'Standard KD exited with code {proc.returncode}. '
            'The real script traceback is printed immediately above.'
        )

d=json.loads((OUT/'metrics.json').read_text()); m=d['dev34']; p=d['protocol']
print(json.dumps(m,indent=2))
assert p['train_participants']==107 and p['dev_participants']==34
assert p['participant_440_excluded'] is True
assert p['threshold']==0.5 and p['threshold_search'] is False
assert p['temperature']==2.0 and p['kd_weight']==0.5
assert p['test_opened'] is False
assert round(float(m['macro_f1']),4)==0.7236
assert round(float(m['depressed_f1']),4)==0.6667
assert round(float(m['auroc']),4)==0.7668
print('PASS: Standard KD DEV-34 reproduced.')